In [16]:
from st_router.paths import get_rootdir
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from dotenv import load_dotenv
import groq
from groq import Groq
from time import sleep


ROOT_DIR = get_rootdir()
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROC = ROOT_DIR / "data" / "processed"
DATA_INTERIM = ROOT_DIR / "data" / "interim"

load_dotenv()

True

In [17]:
bash = pd.read_parquet(DATA_PROC / "bash_windows.parquet")

df = bash.copy()
df = df[df["window_len"] != 0]

# randomly select one row per user - 267 users
df = df.groupby("unique_key").sample(random_state=42)
# from 267 users pick 100 at random
df = df.sample(n=100, random_state=42)
df.head()

,unique_key,timestamp,hostname,cmd,wd,cmd_type,training,gap_s,session_exceed,session,...,prev_2,prev_3,prev_4,prev_5,prev_6,prev_7,prev_8,prev_9,prev_10,window_len
5070,Junior hacker_1_1,2021-02-08 13:06:19.000+00:00,attacker,ssh 10.1.26.100,/home/vagrant,bash-command,Junior hacker,36.000,False,0,...,nmap 10.0.2.0/24,nmap 10.0.2.0,ifconfig,nmap 0.0.0.0,nmap 127.0.0.1,nmap 10.1.26.9,nmap,ls,ls,10
2359,Junior hacker adaptive_2_3,2021-09-09 07:57:20.436+00:00,server,cd Croatia2019,/home/admin/personal,bash-command,Junior hacker adaptive,21.367,False,0,...,ls,cd personal,ls,ssh admin@10.1.26.9,ssh username@admin,nmap 10.1.26.0/24,man nmap,man map,nmap--help,10
11065,Locust 3302_5_26,2021-04-22 14:30:19.910+00:00,attacker,msfconsole,/root,bash-command,Locust 3302,9.080,False,0,...,metasploit,nmap -sV 172.18.1.5,nmap 172.18.1.5,ls,NaN,NaN,NaN,NaN,NaN,5
13830,SQL injection_42_116,2021-01-14 10:27:08.226+00:00,attacker,sqlmap -u server/login.php --forms --tables,/root,bash-command,SQL injection,10.175,False,0,...,sqlmap -h,sqlmap -u server/login.php --forms -search=FLAG,sqlmap -u server/login.php --forms -a,sqlmap -u server/login.php --forms -search,sqlmap -u server/login.php --forms -search,sqlmap -hh,sqlmap -h,sqlmap -u server/login.php --forms -a,sqlmap -u server/login.php --forms --current-...,10
10875,Locust 3302_54_98,2022-02-24 17:17:12.577+00:00,client,cd top-secret,/home/eve,bash-command,Locust 3302,10.669,False,0,...,cd eve,cdi,ls,cd /home,ls,cd ..,ls -la,cd ..,ls -la,10


In [22]:
def create_random_message(row: pd.Series) -> str:
    """Expects a row, returns a message as a string"""
    cmd_list = []
    for i in range(10, 0, -1):
        curr = row[f"prev_{i}"]
        if pd.notna(curr):
            cmd_list.append(curr)
    cmd_str = "\n".join(cmd_list)
    return f"Recent Commands:\n{cmd_str}\ncwd: {row['wd']}"
    
client = Groq()

preds = []
for i, (idx, row) in enumerate(df.iterrows()): 
    sleep(2)
    msg = create_random_message(row)
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are part of a smart-terminal tool whose sole task is predicting the next bash command from a "
                                "context window of past commands, up to 10, and the current working directory. "
                                "Reply with only the next command, on one line, with no explanation.",
                },
                {
                    "role": "user",
                    "content": f"{msg}"
                }

            ],
            model="openai/gpt-oss-20b",
            reasoning_effort="low",
            temperature=0,
        )
        row_dict = {"idx" : idx,
                    "actual" : row["cmd"],
                    "prediction" : chat_completion.choices[0].message.content,
                    "message" : msg,
                    "raw" : chat_completion.to_dict()
                    }
        preds.append(row_dict)
        if (i % 10 == 0): print(f"row {i} processed")
    except groq.APIError as e:
        row_dict = {"idx" : idx,
                    "actual" : row["cmd"],
                    "prediction" : None, 
                    "message" : msg,
                    "raw" : None,
                    "error": str(e),
                    }

        preds.append(row_dict)

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
final = pd.DataFrame(preds)
final.to_json(DATA_INTERIM / "base_rate_gptoss20b.jsonl", orient="records", lines=True)

row 0 processed
row 10 processed
row 20 processed
row 30 processed
row 40 processed
row 50 processed
row 60 processed
row 70 processed
row 80 processed
row 90 processed
